# CDC: Postgres -> Bronze -> Silver (query-based, watermark)
Reads the source DB through **Lakehouse Federation** (`pg_supplychain` foreign catalog). Pulls only rows with
`updated_at > watermark`; the watermark moves only after a successful write (at-least-once; Silver dedups, so re-pulls are harmless).
Limitation: changes between two pulls collapse to the last state, and hard deletes are invisible (log-based CDC would capture both).
**Postgres must be running** for this task.

In [0]:
%run ./_common

In [0]:
NB = "40_cdc_postgres"
import time
SRC = "pg_supplychain.public"
TABLES = ["suppliers", "products", "purchase_orders"]
WM = f"{CAT}.ops.cdc_watermarks"
spark.sql(f"CREATE TABLE IF NOT EXISTS {WM} (table_name STRING, last_updated_at TIMESTAMP, last_run_at TIMESTAMP, rows_pulled BIGINT)")

def pull(t):
    row = spark.table(WM).where(F.col("table_name") == t).select("last_updated_at").first()
    wm = row[0] if row else None
    src = spark.table(f"{SRC}.{t}")
    if wm is not None:
        src = src.where(F.col("updated_at") > F.lit(wm))           # pushed down to Postgres
    batch_id = int(time.time() * 1000)
    (src.withColumn("_cdc_batch_id", F.lit(batch_id)).withColumn("_cdc_pulled_at", F.current_timestamp())
        .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{CAT}.bronze.{t}_cdc"))
    s = (spark.table(f"{CAT}.bronze.{t}_cdc").where(F.col("_cdc_batch_id") == batch_id)
           .agg(F.count("*").alias("n"), F.max("updated_at").alias("mx")).first())
    if s["n"] > 0:
        spark.sql(f"""MERGE INTO {WM} w
            USING (SELECT :t AS table_name, CAST(:mx AS TIMESTAMP) AS last_updated_at,
                          current_timestamp() AS last_run_at, CAST(:n AS BIGINT) AS rows_pulled) s
            ON w.table_name = s.table_name
            WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""",
            args={"t": t, "mx": str(s["mx"]), "n": s["n"]})
    print(f"{t:>16}: watermark {wm} -> pulled {s['n']:,} rows (max updated_at {s['mx']})")

for t in TABLES:
    pull(t)

In [0]:
S, B = f"{CAT}.silver", f"{CAT}.bronze"
spark.sql(f"""
CREATE OR REPLACE TABLE {S}.products_scd2 AS
WITH v AS (SELECT DISTINCT item_id, dept_id, cat_id, supplier_id, unit_cost, status, updated_at FROM {B}.products_cdc)
SELECT *, updated_at AS valid_from,
       LEAD(updated_at) OVER (PARTITION BY item_id ORDER BY updated_at)         AS valid_to,
       LEAD(updated_at) OVER (PARTITION BY item_id ORDER BY updated_at) IS NULL AS is_current
FROM v""")
for t, key in [("purchase_orders", "po_id"), ("suppliers", "supplier_id")]:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.{t} AS
    SELECT * EXCEPT (rn, _cdc_batch_id, _cdc_pulled_at) FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY {key} ORDER BY updated_at DESC, _cdc_pulled_at DESC) AS rn
      FROM {B}.{t}_cdc) WHERE rn = 1""")
print("silver CDC tables rebuilt")

In [0]:
# live reconciliation against the source system
src_po, sil_po = spark.table(f"{SRC}.purchase_orders").count(), table_count(f"{S}.purchase_orders")
dq_check("silver_pos_equal_source", sil_po, sil_po == src_po, f"{src_po} (postgres)")
src_pr = spark.table(f"{SRC}.products").count()
cur_pr = spark.table(f"{S}.products_scd2").where("is_current").count()
dq_check("current_products_equal_source", cur_pr, cur_pr == src_pr, f"{src_pr} (postgres)")
dq_finish(NB)